# PIDNet

PIDNet es una arquitectura para segmetación semántica en tiempo real que emplea el concepto básico de un controlado PID (Proportional-Integral-Derivative). Muchos modelos con gran balance de exactitud y velocidad adecuados para segmentación en tiempo real (BiSeNet, Fast-SCNN, DDRNet, etc) son tipo TBN (Two-Branch-Network). Al analizar esta estrategia desde la perspectiva de un controlador PID, se puede hacer una equivalencia de dicha arquitectura con un controlador PI, el cuál suele tener problemas de sobrepaso (overshoot). A nivel imagen, esto puede traducirse en una rama que aprovecha la información semántica original (P) y otra rama que almacena información contextual de baja frecuencia (I), tal que un modelo de segmentación que usa la fusión de ambas conlleva el riesgo de que los límites de los objetos se vean excesivamente erosionados por los pixeles circundantes y que los objetos pequeños quede eclipsados por los objetos grandes adyacentes. 

Para mitigar eso, se propone adjuntar una tercera rama "derivativa". La rama derivativa de un controlador PID se enfoca en la velocidad de cambio de una señal, permitiendo una mejor sensibilidad a cambios de alta frecuencia. En una imagen, esto representa un mejor enfoque sobre los bordes de los objetos o regiones. Dado que las grietas suelen tener una elevada relación perímetro-área, una red cuya detección de bordes está mejorada, en teoría resulta ideal para una aplicación de segmentación de grietas.

La red propuesta, además utiliza los siguientes módulos específicos para que las ramas interactúen de forma inteligente:
- Pag (Pixel-attention-guided fusion): Este módulo permite que la rama de detalles (P) aprenda selectivamente características semánticas ricas de la rama de contexto (I) sin ser abrumada por ella. Utiliza un mecanismo de atención por píxel para decidir cuánta confianza otorgar a la información de contexto en cada punto.
- PAPPM (Parallel Aggregation Pyramid Pooling Module): Es una versión optimizada del módulo PPM tradicional. En lugar de procesar las escalas de forma secuencial, lo hace de forma paralela para reducir la latencia y mantener la velocidad necesaria en aplicaciones de tiempo real.
- Bag (Boundary-attention-guided fusion): Es el corazón de la integración PID. Utiliza las fronteras detectadas por la rama D para guiar la fusión entre la rama de detalles (P) y la de contexto (I). Básicamente, "obliga" a la red a confiar más en la rama de detalles cuando está cerca de un borde y en la de contexto cuando está dentro de un objeto.

La función de pérdida de la red resulta más compleja. Se compone una suma ponderada de 4 pérdidas que balancean el aprendizaje de bordes y semántica. 

$$ Loss = \lambda_0l_0 + \lambda_1l_1 + \lambda_2l_2 + \lambda_3l_3$$
1. $l0$: Pérdida semántica auxiliar para optimizar toda la red
2. $l1$: Pérdida de entropía cruzada binaria ponderada para la detección de bordes (rama D)
3. $l2$: Pérdida de entropía cruzada estándar para la segmentación final
4. $l3$: Pérdida de entropía cruzada con conciencia de frontera (boundary-awareness), que coordina las tareas de segmentación y detección de bordes

**Adaptación para rama derivativa**: La rama derivativa requiere una supervisión directa mediante mapas de bordes reales para entrenarse correctamente. Dado que el dataset de DeepCrack proporciona únicamente las máscaras binarias grieta/fondo, es necesario generar también las etiquetas de borde. Para ello, se propone generarlas usando un detector de bordes de Canny durante la misma carga del dataset (on-the-fly).

In [ ]:
# ==== Solo para ejecutar en Colab ========

# from google.colab import drive
# import os
# import sys

# # 1. Montar Google Drive
# drive.mount('/content/drive')

# # 2. Definir la ruta a tu proyecto dentro de Drive 
# # (Asegúrate de ajustar esta ruta si guardaste la carpeta en otro lado dentro de tu Drive)
# project_path = '/content/drive/MyDrive/tp_computer_vision_ii'
# notebook_dir = os.path.join(project_path, 'src', 'pidnet')

# # 3. Movernos al directorio donde está el notebook para que las rutas relativas funcionen
# os.chdir(notebook_dir)

# # 4. Agregar la ruta a sys.path para que Python encuentre la carpeta "models"
# if notebook_dir not in sys.path:
#     sys.path.append(notebook_dir)

# print("Directorio actual:", os.getcwd())

In [25]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

from models.pidnet import PIDNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class DeepCrackDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        """
        split: 'train' o 'test'
        """
        self.root_dir = root_dir
        self.img_dir = os.path.join(self.root_dir, f"{split}_img")
        self.lab_dir = os.path.join(self.root_dir, f"{split}_lab")
        self.images = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def _generate_boundary_on_the_fly(self, mask):
        # Habría que jugar un poco con los parámetros de OpenCV para optimizar resultados,
        # pero probablemente lo mejor sería un tratamiento por imagen individual
        # Aplicar Canny para detectar los bordes
        edges = cv2.Canny(mask, 10, 100)

        # Dilatación morfológica para expandir fronteras y conpensar ruido
        kernel = np.ones((3, 3), np.uint8)
        dilated_edges = cv2.dilate(edges, kernel, iterations=1)

        # Normalizar 0-1
        return (dilated_edges > 0).astype(np.float32)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Las máscaras tienen extensión .png, a diferencia de las imágenes
        # que son .jpg
        lab_name = os.path.splitext(img_name)[0] + '.png'
        lab_path = os.path.join(self.lab_dir, lab_name)

        # Cargar imágenes y máscara
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(lab_path, cv2.IMREAD_GRAYSCALE)

        # Redimensionar a un tamaño fijo para todas las imágenes del dataset
        target_size = (512, 512)
        image = cv2.resize(image,target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)

        # Binarizar máscara
        mask = (mask > 127).astype(np.int64)

        # Generar ground truth para la rama derivativa
        boundary = self._generate_boundary_on_the_fly((mask * 255).astype(np.uint8))

        if self.transform:
            image = self.transform(image)
        else:
            # Convertir a tensor si no hay transformación
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return image, torch.tensor(mask, dtype=torch.long), torch.tensor(boundary, dtype=torch.float32)


Es necesario reajustar la arquitectura original, que segmenta en múltiplas clases, para adaptarla a nuestra aplicación en particular que solo tiene dos clases (grieta / no grieta).

In [27]:
def build_custom_pidnet(pretrained_path='PIDNet_M_Cityscapes_test.pt', num_classes=2, variant='S'):
    """
    Construye una PIDNet evitando usar el sistema yacs/cfg del repositorio original y carga
    los pesos pre-entrenados de Cityscapes y los adapta a num_classes.

    Debe descargarse el modelo con los pesos pre-entrenados desde
    https://drive.google.com/drive/folders/0BySIOtxxULinfld0LTcxYndTbFpWNjVpWm9nREU1T3hJUW5IS2otOUJDMmtnZERuODFPVU0?resourcekey=0-nauDQNE1efkunvcg89ZlDA
    y guardar en src\pidnet\pretrained_models, ya que se ignora en el tracking de git por el peso.
    """
    # Configuración de arquitectura siguiendo la base del paper
    if variant == 'S':
        model = PIDNet(m=2, n=3, num_classes=num_classes, planes=32, ppm_planes=96, head_planes=128, augment=True)
    elif variant == 'M':
        model = PIDNet(m=2, n=3, num_classes=num_classes, planes=64, ppm_planes=96, head_planes=128, augment=True)
    elif variant == 'L':
        model = PIDNet(m=3, n=4, num_classes=num_classes, planes=64, ppm_planes=112, head_planes=256, augment=True)
    else:
        raise ValueError("Variante no soportada")

    # Cargar el State Dict pre-entrenado
    print(f"Cargando pesos desde {pretrained_path}...")
    pretrained_dict = torch.load(pretrained_path, map_location='cpu')

    # Manejo de diccionarios si el modelo fue guardado con DataParallel (prefijo 'module.')
    if 'state_dict' in pretrained_dict:
        pretrained_dict = pretrained_dict['state_dict']
    pretrained_dict = {k.replace('module.', '').replace('model.', ''): v for k, v in pretrained_dict.items()}

    model_dict = model.state_dict()

    # Filtrado -> Se conservan solo los pesos cuyas keys existen y cuyas dimensiones coinciden exactamente
    filtered_dict = {
        k: v for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }

    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)

    # print("Checkpoint keys:", list(pretrained_dict.keys())[:10])
    # print("Model keys:", list(model_dict.keys())[:10])

    # Verificación y diagnóstico
    omitidas = set(pretrained_dict.keys()) - set(filtered_dict.keys())
    print(f"\n--- Diagnóstico de Carga ---")
    print(f"Capas pre-entrenadas cargadas con éxito: {len(filtered_dict)}")
    print(f"Capas omitidas (reinicializadas aleatoriamente para {num_classes} clases): {len(omitidas)}")
    for k in sorted(omitidas):
        print(f" -> {k} (Forma original: {pretrained_dict[k].shape})")
        
    return model


<>:8: SyntaxWarning: invalid escape sequence '\p'
<>:8: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Gabriel\AppData\Local\Temp\ipykernel_22556\3601295004.py:8: SyntaxWarning: invalid escape sequence '\p'
  y guardar en src\pidnet\pretrained_models, ya que se ignora en el tracking de git por el peso.


In [ ]:
NUM_CLASSES = 2
VARIANT = "M"
pretrained_path = os.path.join('pretrained_models', 'PIDNet_M_Cityscapes_test.pt')

model = build_custom_pidnet(
    pretrained_path=pretrained_path,
    num_classes=NUM_CLASSES,
    variant=VARIANT
)

Cargando pesos desde pretrained_models\PIDNet_M_Cityscapes_test.pt...

--- Diagnóstico de Carga ---
Capas pre-entrenadas cargadas con éxito: 475
Capas omitidas (reinicializadas aleatoriamente para 2 clases): 30
 -> final_layer.conv2.bias (Forma original: torch.Size([19]))
 -> final_layer.conv2.weight (Forma original: torch.Size([19, 128, 1, 1]))
 -> layer3.3.bn1.bias (Forma original: torch.Size([256]))
 -> layer3.3.bn1.num_batches_tracked (Forma original: torch.Size([]))
 -> layer3.3.bn1.running_mean (Forma original: torch.Size([256]))
 -> layer3.3.bn1.running_var (Forma original: torch.Size([256]))
 -> layer3.3.bn1.weight (Forma original: torch.Size([256]))
 -> layer3.3.bn2.bias (Forma original: torch.Size([256]))
 -> layer3.3.bn2.num_batches_tracked (Forma original: torch.Size([]))
 -> layer3.3.bn2.running_mean (Forma original: torch.Size([256]))
 -> layer3.3.bn2.running_var (Forma original: torch.Size([256]))
 -> layer3.3.bn2.weight (Forma original: torch.Size([256]))
 -> layer3.3.c

In [29]:
# import albumentations as A
# from albumentations.pytorch import ToTensorV2

# # Definir las transformaciones (Escalado aleatorio, Recorte y Normalización ImageNet)
# train_transform = A.Compose([
#     A.RandomScale(scale_limit=(0.5, 2.0), p=1.0),
#     A.RandomCrop(height=512, width=512), # Ajusta al tamaño de tu GPU
#     A.HorizontalFlip(p=0.5),
#     A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ToTensorV2(),
# ])
train_transform = None
if device == 'cuda':
    num_workers = 2
else:
    num_workers = 0

# Instanciar el Dataset (usando la clase DeepCrackDataset adaptada para albumentations)
train_dataset = DeepCrackDataset(root_dir='../../datasets/dataset_1000', split='train', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=num_workers, drop_last=True)

val_dataset = DeepCrackDataset(root_dir='../../datasets/dataset_1000', split='test', transform=train_transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=num_workers)

In [30]:
# class BoundaryAwareCrossEntropy(nn.Module):
#     def __init__(self, ignore_index=255):
#         super().__init__()
#         self.ignore_index = ignore_index
#         self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)

#     def forward(self, pred, boundary_logits, target):
#         # pred: [B, C, H, W]
#         # boundary_logits: [B, 1, H, W] o [B, H, W]
#         # target: [B, H, W]

#         if boundary_logits.dim() == 4:
#             boundary_logits = boundary_logits.squeeze(1)

#         if boundary_logits.shape[-2:] != target.shape[-2:]:
#             boundary_logits = F.interpolate(
#                 boundary_logits.unsqueeze(1),
#                 size=target.shape[-2:],
#                 mode='bilinear',
#                 align_corners=True
#             ).squeeze(1)

#         prob = torch.sigmoid(boundary_logits)
#         mask = prob > 0.8

#         new_target = target.clone()
#         new_target[~mask] = self.ignore_index

#         return self.ce(pred, new_target)


# def weighted_bce_with_logits(logits, target):
#     logits = logits.view(-1)
#     target = target.float().view(-1)

#     pos = (target == 1)
#     neg = (target == 0)

#     weight = torch.zeros_like(logits)
#     pos_num = pos.sum()
#     neg_num = neg.sum()
#     total = pos_num + neg_num

#     weight[pos] = neg_num / total
#     weight[neg] = pos_num / total

#     return F.binary_cross_entropy_with_logits(logits, target, weight=weight, reduction='mean')

class BoundaryAwareCrossEntropy(nn.Module):
    def __init__(self, gamma=1.0):
        super(BoundaryAwareCrossEntropy, self).__init__()
        self.gamma = gamma

    def forward(self, semantic_preds, boundary_preds, targets):
        """
        semantic_preds: Probabilidades Softmax (s_head2) [B, C, H, W]
        boundary_preds: Activaciones Sigmoide (d_head) [B, 1, H, W]
        targets: Etiquetas manuales [B, H, W]
        """
        # Calcular entropía cruzada estándar sin reducción
        ce_loss = F.cross_entropy(semantic_preds, targets, reduction='none')
        
        # Escalar la pérdida espacialmente con las predicciones de frontera
        # (1 + gamma * p_boundary)
        boundary_weight = 1.0 + (self.gamma * boundary_preds.squeeze(1))
        
        # Aplicar el castigo ponderado
        weighted_ce = ce_loss * boundary_weight
        
        return weighted_ce.mean()

In [31]:
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

# 1. Definición de las funciones de pérdida
criterion_semantic = nn.CrossEntropyLoss(ignore_index=255) # l_0 y l_2
criterion_boundary = nn.BCEWithLogitsLoss()                # l_1
criterion_boundary_aware = BoundaryAwareCrossEntropy()     # l_3 (La que definimos en el paso anterior)

# 2. Optimizador (SGD con Nesterov)
optimizer = optim.SGD(
    model.parameters(), 
    lr=1e-3, 
    momentum=0.9, 
    weight_decay=5e-4, 
    nesterov=True
)

# 3. Scheduler 'poly'
max_epochs = 100
power = 0.9
def poly_lr_scheduler(epoch):
    return (1 - epoch / max_epochs) ** power

scheduler = LambdaLR(optimizer, lr_lambda=poly_lr_scheduler)

In [ ]:
def validate(
    model,
    val_loader,
    criterion_semantic,
    criterion_boundary,
    criterion_boundary_aware,
    device
):
    """
    Calcluar la pérdida de validación.
    """
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks, boundaries in val_loader:
            
            images = images.to(device)
            masks = masks.to(device)
            boundaries = boundaries.to(device).unsqueeze(1)

            # Forward pass
            outputs = model(images)
            pred_p = outputs[0]
            pred_final = outputs[1]
            pred_d = outputs[2]

            # Interpolar al tamaño original
            h, w = masks.shape[1], masks.shape[2]
            pred_p = F.interpolate(pred_p, size=(h, w), mode='bilinear', align_corners=True)
            pred_final = F.interpolate(pred_final, size=(h, w), mode='bilinear', align_corners=True)
            pred_d = F.interpolate(pred_d, size=(h, w), mode='bilinear', align_corners=True)

            # Cálcular pérdidas
            loss_0 = criterion_semantic(pred_p, masks)
            loss_1 = criterion_boundary(pred_d, boundaries)
            loss_2 = criterion_semantic(pred_final, masks)
            pred_d_sigmoid = torch.sigmoid(pred_d)
            loss_3 = criterion_boundary_aware(pred_final, pred_d_sigmoid, masks)

            # Agregación
            loss_total = (0.4 * loss_0) + (20.0 * loss_1) + (1.0 * loss_2) + (1.0 * loss_3)
            val_loss += loss_total.item()

    return val_loss / len(val_loader)

In [ ]:
model.to(device)

best_val_loss = float('inf')
patience = 8
patience_counter = 0

train_loss_hist = []
val_loss_hist = []
val_miou_hist = []


for epoch in range(max_epochs):
    # ======= ENTRENAMIENTO =======
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (images, masks, boundaries) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        boundaries = boundaries.to(device).unsqueeze(1) # [B, 1, H, W]
        
        optimizer.zero_grad()
        
        # 1. Forward pass: Obtienes los tensores a 1/8 de resolución
        outputs = model(images)
        pred_p = outputs[0]
        pred_final = outputs[1]
        pred_d = outputs[2]
        
        # --- LA CORRECCIÓN CLAVE: Interpolar al tamaño original ---
        # En el repo original esta interpolación se maneja de forma "invisible"
        # directamente sobre las funciones de pérdida personalizadas. 
        # 
        # Se extrae H y W de la máscara ground-truth (que tiene el tamaño real)
        h, w = masks.shape[1], masks.shape[2]
        
        # Sobremuestreamos usando interpolación bilineal (recomendado para segmentación)
        pred_p = F.interpolate(pred_p, size=(h, w), mode='bilinear', align_corners=True)
        pred_final = F.interpolate(pred_final, size=(h, w), mode='bilinear', align_corners=True)
        pred_d = F.interpolate(pred_d, size=(h, w), mode='bilinear', align_corners=True)
        # ---------------------------------------------------------
        
        # 2. Calcular pérdidas individuales (ahora los tensores coinciden en dimensiones)
        loss_0 = criterion_semantic(pred_p, masks)
        loss_1 = criterion_boundary(pred_d, boundaries)
        loss_2 = criterion_semantic(pred_final, masks)
        
        # Probabilidades sigmoides requeridas por la pérdida l_3
        pred_d_sigmoid = torch.sigmoid(pred_d)
        loss_3 = criterion_boundary_aware(pred_final, pred_d_sigmoid, masks)
        
        # 3. Agregación total (Valores de lambdas recomendado por los autores)
        loss_total = (0.4 * loss_0) + (20.0 * loss_1) + (1.0 * loss_2) + (1.0 * loss_3)
        
        # 4. Backward pass
        loss_total.backward()
        optimizer.step()
        
        epoch_loss += loss_total.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch}/{max_epochs}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss_total.item():.4f}")
    
    # ====== VALIDACIÓN ======
    train_avg_loss = epoch_loss / len(train_loader)
    val_avg_loss = validate(
        model, val_loader, criterion_semantic, criterion_boundary,
        criterion_boundary_aware, device
    )
    
    print(f"--- Fin de Epoch {epoch} ---")
    print(f"Train Loss: {train_avg_loss:.4f} | Val Loss: {val_avg_loss:.4f}")

    # Se guarda el mejor modelo y se actualiza el contador para early stop
    if val_avg_loss < best_val_loss:
        best_val_loss = val_avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_avg_loss,
        }, 'best_pidnet.pth')
        print(f"Mejor modelo guardado (Val Loss: {val_avg_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping activado en epoch {epoch}")
        break
            
    # Actualizar la tasa de aprendizaje al final de la época
    scheduler.step()
    print(f"--- Fin de Epoch {epoch} | Loss Promedio: {epoch_loss/len(train_loader):.4f} ---")

Epoch [0/100] Batch [0/37] Loss: 14.0826


KeyboardInterrupt: 